# SVAMITVA SAM2 - Optimized DGX Training

- **Staged Training Approach**: Sequentially adjusts learning rates and backbone freezing for maximum accuracy.
- **Single Best.pt**: Automatically tracks and updates the best model state (`best.pt`) across all training phases.
### Features:
- **Smart GPU Choice**: Automatically selects the GPU with the most free VRAM.
- **Layer-by-Layer Training**: Sequentially trains different components to ensure stable convergence.
- **Single Best.pt**: Automatically tracks and updates the best model state across all training phases.
- **Perfect Edges**: Enabled Lovász Loss and Refinement Layers for sharp segmentation.


## 1. Environment & GPU Selection
We search for the GPU with the most free memory to avoid conflicts with other users on the DGX.

In [ ]:
import torch
import os
import subprocess

def get_best_gpu():
    """Find the GPU with the most free memory using nvidia-smi."""
    try:
        cmd = "nvidia-smi --query-gpu=memory.free --format=csv,nounit,noheader"
        free_memory = [int(x) for x in subprocess.check_output(cmd.split()).decode('ascii').split()]
        best_gpu = free_memory.index(max(free_memory))
        print(f"✅ Selected GPU {best_gpu} with {free_memory[best_gpu]}MB free VRAM")
        return best_gpu
    except Exception as e:
        print(f"⚠️ Could not query nvidia-smi, defaulting to GPU 0: {e}")
        return 0

best_gpu = get_best_gpu()
os.environ["CUDA_VISIBLE_DEVICES"] = str(best_gpu)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Current Device: {device}")

## 2. Configuration
Set your dataset paths and basic hyperparameters.

In [ ]:
DATA_DIRS = [
    "data/MAP1",  # Update these to your dataset paths
    "data/MAP2"
]

EXPERIMENT_NAME = "perfect_edges_staged"
BATCH_SIZE = 16
LR = 3e-4
CHECKPOINT_DIR = "check"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

## 3. Phase 1: Initial Warm-up & Base Training
Train all task heads with the SAM2 backbone partially frozen to establish strong initial spatial features.

In [ ]:
from train import main
import sys

print("🚀 Training Buildings (Phase 1)")
sys.argv = [
    'train.py',
    '--epochs', '50',
    '--name', EXPERIMENT_NAME,
    '--batch_size', str(BATCH_SIZE),
    '--epochs', '20',
    '--lr', str(LR),
    '--checkpoint_dir', CHECKPOINT_DIR
]
main()

## 4. Phase 2: Intermediate Refinement
Resuming from Phase 1. We decrease the learning rate to allow the refinement layers to converge without overshooting.

In [ ]:
print("🚀 Intermediate Refinement (Phase 2)")
sys.argv = [
    'train.py',
    '--train_dirs'] + DATA_DIRS + [
    '--name', EXPERIMENT_NAME,
    '--batch_size', str(BATCH_SIZE),
    '--epochs', '20',
    '--epochs', '50',
    '--resume', f'{CHECKPOINT_DIR}/best.pt',
    '--checkpoint_dir', CHECKPOINT_DIR
]
main()

## 5. Phase 3: Final End-to-End Fine-tuning
Unfreeze the entire SAM2 backbone and train across all tasks with a minimal learning rate to extract the finest details and perfect edges.

In [ ]:
print("🚀 Final Staged Fine-tuning (Phase 3)")
sys.argv = [
    'train.py',
    '--train_dirs'] + DATA_DIRS + [
    '--name', EXPERIMENT_NAME,
    '--batch_size', str(BATCH_SIZE),
    '--epochs', '50',
    '--epochs', '100',
    '--lr', str(LR / 10),
    '--resume', f'{CHECKPOINT_DIR}/best.pt',
    '--checkpoint_dir', CHECKPOINT_DIR
]
main()

## 6. Summary
Your final optimized model is in `check/best.pt`.